In [1]:
import os
import re
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm


def get_config(dataset: str) -> dict:
    if dataset == "mall":
        return {
            "label_file": "./mall/labels.csv",
            "data_dir": "./mall",
            "process_label_file": "../mall/label_process.csv",
            "max_span": 200,
        }
    elif dataset == "train-ticket":
        return {
            "label_file": "./train-ticket/labels.csv",
            "data_dir": "./train-ticket",
            "process_label_file": "../train-ticket/label_process.csv",
            "max_span": 200,
        }
    else:
        raise ValueError("dataset must be 'mall' or 'train-ticket'")


class LabelProcessor:
    def __init__(self, config: dict):
        self.config = config
        self.label_file = config["label_file"]
        self.data_dir = config["data_dir"]
        self.process_label_file = config["process_label_file"]
        self.max_span = config["max_span"]

        self.label_df = None
        self.fault_2_id = None

    def load_labels(self) -> pd.DataFrame:
        temp_label_df = pd.read_csv(self.label_file)
        '''
             start_time    end_time                                    service_name  \
            0  1711596900  1711597020                                   ['mall-gateway']   
            1  1711606200  1711606440                                   ['mall-gateway']   

            
                                         operation_name  excluded_operation_name  \
            0                                       NaN                      NaN   
            1  ['POST /components/api/v1/mall/product']                      NaN   

            
              fault_type                          fault  
            0  Container  Container network packet loss  
            1    Service        Service interface delay  
        '''
        self.label_df = pd.read_csv(self.label_file)
        return self.label_df

    def build_fault_mapping(self) -> dict:
        if self.label_df is None:
            self.load_labels()
        self.fault_2_id = {
            fault: idx for idx, fault in enumerate(set(self.label_df["fault"].to_list()))
        }
        return self.fault_2_id

    def get_trace_dirs_by_date(self, date_string: str) -> list[str]:
        if date_string == "2024-06-27":
            return [
                f"{self.data_dir}/{date_string}/trace/",
                f"{self.data_dir}/{date_string}-1/trace/",
            ]
        return [f"{self.data_dir}/{date_string}/trace/"]

    def match_trace_files(self, timestamp_s: int, timestamp_e: int) -> list[str]:
        dt_time = datetime.fromtimestamp(timestamp_s)
        date_string = dt_time.strftime("%Y-%m-%d")
        data_file_dir_list = self.get_trace_dirs_by_date(date_string)

        matched_files = []
        for data_file_dir in data_file_dir_list:
            for _, _, files in os.walk(data_file_dir):
                for file_d in files:
                    if ".parquet" not in file_d:
                        continue

                    # trace记录的开始时间戳和结束时间戳
                    timestamps = re.findall(r"\d+", file_d)
                    if len(timestamps) < 2:
                        continue

                    timestamp1, timestamp2 = map(int, timestamps[:2])

                    if timestamp1 <= timestamp_s <= timestamp2:
                        matched_files.append(
                            f"{data_file_dir}trace_info_{timestamp1}_{timestamp2}.parquet"
                        )
                    elif timestamp1 <= timestamp_e <= timestamp2:
                        matched_files.append(
                            f"{data_file_dir}trace_info_{timestamp1}_{timestamp2}.parquet"
                        )

        return list(set(matched_files))

    def attach_files_to_labels(self) -> pd.DataFrame:
        if self.label_df is None:
            self.load_labels()
        if self.fault_2_id is None:
            self.build_fault_mapping()

        file_list = []
        fault_id_list = []

        for _, row in tqdm(
            self.label_df.iterrows(),
            total=len(self.label_df),
            postfix="Link faults with files",
        ):
            timestamp_s = row["start_time"]
            timestamp_e = row["end_time"]
            fault = row["fault"]

            # 收集故障注入开始时间/结束时间在trace收集范围内(即有重叠)的trace文件
            matched_files = self.match_trace_files(timestamp_s, timestamp_e)
            file_list.append(matched_files)
            fault_id_list.append(self.fault_2_id[fault])

        self.label_df["file_name"] = file_list
        self.label_df["fault_id"] = fault_id_list
        # 原始self.label_df
        '''
             start_time    end_time                                    service_name  \
            0  1711596900  1711597020                                   ['mall-gateway']   
            1  1711606200  1711606440                                   ['mall-gateway']   

            
                                         operation_name  excluded_operation_name  \
            0                                       NaN                      NaN   
            1  ['POST /components/api/v1/mall/product']                      NaN   

            
              fault_type                          fault  
            0  Container  Container network packet loss  
            1    Service        Service interface delay  
        '''
        # 给每一条fault记录添加对应的trace文件和fault_id
        return self.label_df

    @staticmethod
    def build_trace_filter_condition(df_merge: pd.DataFrame, row: pd.Series) -> pd.Series: 
        # df_merge合并的trace
        # row为故障注入记录
        cond = (
            (df_merge["Timestamp"] >= row["start_time"] * 1000) &
            (df_merge["Timestamp"] <= row["end_time"] * 1000)
        )

        # 保留故障输入记录发生微服务的追踪
        cond &= df_merge["ServiceName"].isin(eval(row["service_name"]))

        if type(row["excluded_operation_name"]) != type(np.nan):
            cond &= ~(df_merge["OperationName"].isin(eval(row["excluded_operation_name"])))
        elif type(row["operation_name"]) != type(np.nan):
            cond &= df_merge["OperationName"].isin(eval(row["operation_name"]))

        if row["fault"] == "Service error":
            cond &= (df_merge["ResultCode"] == "1")

        return cond

    def extract_trace_ids_for_row(self, row: pd.Series) -> tuple[list, list]:
        file_name_list = row["file_name"]
        df_list = []

        for file_url in file_name_list:
            if os.path.exists(file_url):
                df_list.append(pd.read_parquet(file_url))
                '''
                 Duration  HaveStack LogEventList                           OperationName  \
                0        29      False           []       /components/api/v1/mall/user_info   
                1        28      False           []                 MallController.userInfo   
                
                       ParentSpanId ResultCode RpcId  RpcType   ServiceIp       ServiceName  \
                0                 0         -1              0  10.0.0.176      mall-gateway   
                1  1aaf1265717fa2db         -1              0  10.0.0.176      mall-gateway   
                
                             SpanId                                       TagEntryList  \
                0  1aaf1265717fa2db  [{"Key":"app","Value":"mall-gateway"},{"Key":"...   
                1  5ec8742aec5925dd  [{"Key":"app","Value":"mall-gateway"},{"Key":"...   
                
                       Timestamp                           TraceID  
                0  1711595187228  4d1f3abd3e3683e34d6acd44c1ca144f  
                1  1711595187229  4d1f3abd3e3683e34d6acd44c1ca144f  

                '''

        if len(df_list) == 0:
            return [], []

        df_merge = pd.concat(df_list)
        df_group = df_merge.groupby("TraceID")
        trace_id_2_df_len = {trace_id: len(df) for trace_id, df in df_group}

        # 筛选得到与row所对应的故障注入记录相关的span
        cond = self.build_trace_filter_condition(df_merge, row)
        df_merge = df_merge[cond]


        # 过滤掉太长的trace
        trace_ids = list(set(df_merge["TraceID"].to_list()))
        kept_trace_ids = [
            trace_id for trace_id in trace_ids
            if trace_id_2_df_len[trace_id] < self.max_span
        ]
        span_lens = [trace_id_2_df_len[trace_id] for trace_id in kept_trace_ids]

        '''
            (
                ["trace1", "trace2", ...],
                [len1, len2, ...]
            )
        '''
        return kept_trace_ids, span_lens

    def attach_trace_ids_to_labels(self) -> pd.DataFrame:
        if self.label_df is None:
            raise ValueError("Please run attach_files_to_labels() first.")

        trace_id_list = []
        span_len_list = []

        for _, row in tqdm(
            self.label_df.iterrows(),
            total=len(self.label_df),
            postfix="Link faults with trace ids",
        ):
            trace_ids, span_lens = self.extract_trace_ids_for_row(row)
            trace_id_list.append(trace_ids) # 每条 fault 对应的 trace_id
            span_len_list.append(span_lens) # 每个 trace 的长度

        self.label_df["trace_id_list"] = trace_id_list # 每条记录对应的多个 trace（样本）
        self.label_df["count"] = [len(trace_ids) for trace_ids in trace_id_list] # 每个fault对应多少个trace
        self.label_df["span_len"] = span_len_list # 每个trace的长度（span数量）
        return self.label_df

    def save(self) -> None:
        if self.label_df is None:
            raise ValueError("No processed dataframe to save.")
        '''
            fault            start_time    end_time     file_name \                                  
            CPU overload     (s)             (s)      "['./trace/file1.parquet', './trace/file2.parquet']"
            Service error    (s)             (s)

            fault_id    trace_id_list                       count            span_len
            0           "['traceA', 'traceB', 'traceC']"    trace_id 的数量   每个 trace 的长度（span数量）
        '''
        self.label_df.to_csv(self.process_label_file, index=False)
        print(f"save process label file to: {self.process_label_file}")

    def run(self) -> pd.DataFrame:
        self.load_labels()
        self.build_fault_mapping()
        self.attach_files_to_labels()
        self.attach_trace_ids_to_labels()
        self.save()
        return self.label_df

In [4]:
dataset = 'train-ticket'   # 'mall'或者'train-ticket'
config = get_config(dataset)
processor = LabelProcessor(config)
result_df = processor.run()

100%|██████████| 31/31 [00:30<00:00,  1.00it/s, Link faults with trace ids]

save process label file to: ../train-ticket/label_process.csv
